In [12]:
import cv2
import torch
import sys
import numpy as np
import pandas as pd
from pathlib import Path
from omegaconf import OmegaConf
from collections import defaultdict, Counter
from ultralytics import YOLO
import matplotlib.pyplot as plt
import importlib

sys.path.insert(0, str(Path.cwd() / "MAIN_MODULE" / "src"))
sys.path.insert(0, str(Path.cwd() / "DEPARTMENT_CLASSIFICATION" / "train_model"))
sys.path.insert(0, str(Path.cwd() / "IMG PREPROCESSING"))
sys.path.insert(0, str(Path.cwd() / "CROP_QUALITY_CLASSIFICATION"))

from quality_classifier.predict import quality_classifier
from color_classifier import process_dataset
import crop_extraction
importlib.reload(crop_extraction)
from crop_extraction import CropCandidate, CropScorer
from predict_single import DepartmentPredictor
import distortion
importlib.reload(distortion)
from distortion import DistortionCorrector, CAM_SETTINGS, CAM_DISTORT_COEFFS



sys.path.insert(0, str(Path.cwd() / "VLM_MODULE"))
from detect import load_vlm_model, vlm_predict_crops
sys.path.insert(0, str(Path.cwd() / "LLMTEXT"))
from product_matcher import find_top5_matches

In [13]:
config = OmegaConf.load('params.yaml')
root = Path.cwd()
video_folder = root / config.main_extraction.input_folder
yolo_path = root / config.main_extraction.model_path
dept_model_path = root / config.department_classifier.model_path
class_names_path = root / "DEPARTMENT_CLASSIFICATION/train_model/models/class_names.json"

In [14]:
video_extensions = {".mp4", ".avi", ".mov", ".mkv", ".wmv", ".webm"}
videos = sorted(p for p in video_folder.iterdir()
                if p.suffix.lower() in video_extensions and not p.name.startswith("~"))
video_path = videos[0]
print(f"Видео: {video_path.name}")

Видео: 25_12-20.mp4


In [15]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
yolo = YOLO(str(yolo_path)).to(device)
crop_scorer = CropScorer(
    config.main_extraction.min_crop_width,
    config.main_extraction.min_crop_height,
    config.main_extraction.sharpness_threshold,
)

def _predict_np(self, img, top_k=3):
    if img.ndim == 2:
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
    elif img.shape[2] == 4:
        img = cv2.cvtColor(img, cv2.COLOR_BGRA2RGB)
    elif img.shape[2] == 3:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    t = self.transform(image=img)
    x = t['image'].unsqueeze(0).to(self.device)
    with torch.no_grad():
        p = torch.softmax(self.model(x), dim=1)[0]
    top = torch.topk(p, top_k)
    return [(self.class_names[i.item()], v.item() * 100) for v, i in zip(top.values, top.indices)]

DepartmentPredictor.predict_np = _predict_np
classifier = DepartmentPredictor(str(dept_model_path), str(class_names_path))

# Инициализация корректора дисторсии
corrector = DistortionCorrector(CAM_SETTINGS, CAM_DISTORT_COEFFS)
print(f"Distortion corrector initialized. ROI: {corrector.roi}")

Создана efficientnet-b0 со случайными весами
Модель загружена: efficientnet-b0
Классов: 15
Устройство: cuda


TypeError: DistortionCorrector.__init__() got an unexpected keyword argument 'alpha'

In [ ]:
cap = cv2.VideoCapture(str(video_path))
fps = cap.get(cv2.CAP_PROP_FPS)
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
cap.release()
print(f"{total} frames, {fps:.2f} FPS")

In [ ]:
seg_size = total // 5
half_window = 15
step = 5
segment_frames = []
for i in range(5):
    mid = i * seg_size + seg_size // 2
    start = max(0, mid - half_window)
    end = min(total - 1, mid + half_window)
    segment_frames.append(list(range(start, end + 1, step)))
for i, frames in enumerate(segment_frames):
    print(f"  Сегмент {i+1}: {len(frames)} кадров ({frames[0]/fps:.1f}с - {frames[-1]/fps:.1f}с)")

In [ ]:
best: dict[int, list[CropCandidate]] = defaultdict(list)
frame_to_seg = {}
for sid, frames in enumerate(segment_frames):
    for f in frames:
        frame_to_seg[f] = sid
seg_preds = [[] for _ in range(5)]

# Для визуализации оригинальных bbox по таймстампам
original_bbox_by_timestamp = {}

cap = cv2.VideoCapture(str(video_path))
fi = -1
while True:
    ok, fr = cap.read()
    if not ok:
        break
    fi += 1

    timestamp_ms = int(fi / fps * 1000)
    h_orig, w_orig = fr.shape[:2]

    # ============================================
    # 1. Дисторсия (как в distortion_with_correction.ipynb)
    # ============================================
    fr_corrected = corrector.undistort_frame(fr.copy())

    # ============================================
    # 2. Поворот (и оригинала для визуализации, и скорректированного)
    # ============================================
    if config.main_extraction.rotate_frames:
        fr_orig_rotated = cv2.rotate(fr, cv2.ROTATE_90_COUNTERCLOCKWISE)
        fr_corrected = cv2.rotate(fr_corrected, cv2.ROTATE_90_COUNTERCLOCKWISE)
    else:
        fr_orig_rotated = fr

    h_corr, w_corr = fr_corrected.shape[:2]

    # ============================================
    # 3. YOLO на скорректированном фрейме
    # ============================================
    res = yolo.track(source=fr_corrected, persist=True,
                     tracker=config.main_extraction.tracker_config,
                     conf=config.main_extraction.conf_threshold,
                     iou=config.main_extraction.iou_threshold, verbose=False)[0]

    boxes_data = []
    if res.boxes is not None and res.boxes.id is not None:
        for box, conf, tid in zip(res.boxes.xyxy.cpu().numpy(),
                                   res.boxes.conf.cpu().numpy(),
                                   res.boxes.id.cpu().numpy().astype(int)):
            x1, y1, x2, y2 = map(int, box)
            x1, y1, x2, y2 = max(0, x1), max(0, y1), min(w_corr, x2), min(h_corr, y2)
            boxes_data.append((tid, [x1, y1, x2, y2], float(conf)))

    # ============================================
    # 4. Оригинальные bbox для визуализации
    #    (масштабирование из скорректированного в оригинальный повёрнутый)
    # ============================================
    orig_boxes_for_viz = []
    for tid, bbox_corr, conf in boxes_data:
        x1_c, y1_c, x2_c, y2_c = bbox_corr
        # Масштабирование
        scale_x = w_orig / w_corr
        scale_y = h_orig / h_corr
        x1_o = int(x1_c * scale_x)
        y1_o = int(y1_c * scale_y)
        x2_o = int(x2_c * scale_x)
        y2_o = int(y2_c * scale_y)
        orig_boxes_for_viz.append((tid, [x1_o, y1_o, x2_o, y2_o]))

    original_bbox_by_timestamp[timestamp_ms] = orig_boxes_for_viz

    # ============================================
    # 5. Сохраняем кропы
    # ============================================
    for tid, bbox_corr, conf in boxes_data:
        x1, y1, x2, y2 = bbox_corr
        crop = fr_corrected[y1:y2, x1:x2]

        score = crop_scorer.compute_score(crop, conf)
        if score is None:
            continue

        # Находим оригинальный bbox для этого track_id
        bbox_orig = None
        for viz_tid, viz_bbox in orig_boxes_for_viz:
            if viz_tid == tid:
                bbox_orig = viz_bbox
                break

        c = CropCandidate(
            score=score,
            crop=crop.copy(),
            frame_index=fi,
            confidence=conf,
            bbox=bbox_corr,
            bbox_original=bbox_orig
        )
        best[tid].append(c)
        best[tid].sort(key=lambda x: x.score, reverse=True)
        best[tid] = best[tid][:config.main_extraction.top_k]

    # Department classification (на скорректированном повёрнутом)
    if fi in frame_to_seg:
        sid = frame_to_seg[fi]
        dept, prob = classifier.predict_np(fr_corrected)[0]
        seg_preds[sid].append({'frame': fi, 'time': fi / fps, 'department': dept, 'prob': prob})

    if fi % 500 == 0:
        print(f"Frame {fi}/{total}, tracks: {len(best)}")

cap.release()
print(f"Done. Tracks: {len(best)}, crops: {sum(len(v) for v in best.values())}")

In [ ]:
# Сбор результатов по предсказанию отдела
rows_seg = []
for sid, preds in enumerate(seg_preds):
    for p in preds:
        rows_seg.append({
            'segment': sid + 1,
            'frame': p['frame'],
            'time_sec': round(p['time'], 1),
            'department': p['department'],
            'probability': round(p['prob'], 1),
        })

department = pd.DataFrame(rows_seg)

In [ ]:
department

In [ ]:
rows_crops = []
for track_id, candidates in best.items():
    for rank, c in enumerate(candidates):
        rows_crops.append({
            'filename': video_path.name,
            'SYS_track_id': track_id,
            'SYS_rank': rank + 1,
            'SYS_score': round(c.score, 1),
            'SYS_confidence': round(c.confidence, 3),
            'product_name': None, 'price_default': None, 'price_card': None,
            'price_discount': None, 'barcode': None, 'discount_amount': None,
            'id_sku': None, 'print_datetime': None, 'code': None,
            'additional_info': None, 'color': None, 'special_symbols': None,
            'frame_timestamp': int(c.frame_index / fps * 1000),
            
            # Координаты в скорректированном фрейме
            'x_min_corrected': c.bbox[0], 'y_min_corrected': c.bbox[1],
            'x_max_corrected': c.bbox[2], 'y_max_corrected': c.bbox[3],
            
            # Координаты в оригинальном фрейме (из ПРОХОДА 1)
            'x_min_original': c.bbox_original[0] if c.bbox_original else None,
            'y_min_original': c.bbox_original[1] if c.bbox_original else None,
            'x_max_original': c.bbox_original[2] if c.bbox_original else None,
            'y_max_original': c.bbox_original[3] if c.bbox_original else None,
            
            'qr_code_barcode': None, 'price1_qr': None, 'price2_qr': None,
            'price3_qr': None, 'price4_qr': None,
            'wholesale_level_1_count': None, 'wholesale_level_1_price': None,
            'wholesale_level_2_count': None, 'wholesale_level_2_price': None,
            'action_price_qr': None, 'action_code_qr': None,
        })

df_crops = pd.DataFrame(rows_crops)

In [ ]:
crops_for_color = [(track_id, candidate.crop) 
                   for track_id, candidates in best.items() 
                   for candidate in candidates]
# Классификация цветов
color_results = process_dataset(crops_for_color)
# Добавление в DataFrame
df_crops['color'] = df_crops['SYS_track_id'].map(color_results)

# === ВИЗУАЛИЗАЦИЯ: Кропы по цветам ===
df_color_check = pd.DataFrame([
    {
        'track_id': track_id,
        'color': color_results[track_id],
        'crop': crop_image
    }
    for track_id, crop_image in crops_for_color
])

# Группируем по цветам
df_red = df_color_check[df_color_check['color'] == 'red']
df_yellow = df_color_check[df_color_check['color'] == 'yellow']
df_white = df_color_check[df_color_check['color'] == 'white']

n_red, n_yellow, n_white = len(df_red), len(df_yellow), len(df_white)
n_max = max(n_red, n_yellow, n_white, 1)

if n_max > 0:
    fig, axes = plt.subplots(n_max, 3, figsize=(15, 5 * n_max))
    
    # Красные
    for i, (_, row) in enumerate(df_red.iterrows()):
        if i < n_max:
            axes[i, 0].imshow(cv2.cvtColor(row['crop'], cv2.COLOR_BGR2RGB))
            axes[i, 0].set_title(f"Красный: {row['track_id']}", fontsize=9)
            axes[i, 0].axis('off')
    
    # Жёлтые
    for i, (_, row) in enumerate(df_yellow.iterrows()):
        if i < n_max:
            axes[i, 1].imshow(cv2.cvtColor(row['crop'], cv2.COLOR_BGR2RGB))
            axes[i, 1].set_title(f"Жёлтый: {row['track_id']}", fontsize=9)
            axes[i, 1].axis('off')
    
    # Белые
    for i, (_, row) in enumerate(df_white.iterrows()):
        if i < n_max:
            axes[i, 2].imshow(cv2.cvtColor(row['crop'], cv2.COLOR_BGR2RGB))
            axes[i, 2].set_title(f"Белый: {row['track_id']}", fontsize=9)
            axes[i, 2].axis('off')
    
    # Скрыть пустые ячейки
    for i in range(n_red, n_max):
        axes[i, 0].axis('off')
    for i in range(n_yellow, n_max):
        axes[i, 1].axis('off')
    for i in range(n_white, n_max):
        axes[i, 2].axis('off')
    
    plt.tight_layout()
    plt.show()

print(f"Красные: {n_red}, Жёлтые: {n_yellow}, Белые: {n_white}")

In [ ]:
# Сбор кропов для классификации качества
crops_for_quality = [(track_id, candidate.crop) 
                     for track_id, candidates in best.items() 
                     for candidate in candidates]

# Классификация качества (мусор/нет)
quality_model_path = root / config.quality_classifier.model_path
trash_map, confidence_map = quality_classifier(str(quality_model_path), crops_for_quality)

# Добавление в DataFrame
df_crops['SYS_trash'] = df_crops['SYS_track_id'].map(trash_map)

# === ВИЗУАЛИЗАЦИЯ: Кропы (мусор/хорошие) ===
df_check = pd.DataFrame([
    {
        'track_id': track_id,
        'is_trash': trash_map[track_id],
        'confidence': confidence_map[track_id],
        'crop': crop_image
    }
    for track_id, crop_image in crops_for_quality
])

df_bad = df_check[df_check['is_trash'] == True].sort_values('confidence', ascending=False)
df_good = df_check[df_check['is_trash'] == False].sort_values('confidence', ascending=True)

n_bad, n_good = len(df_bad), len(df_good)
n_total = max(n_bad, n_good)

if n_total > 0:
    if n_total == 1:
        fig, axes = plt.subplots(1, 2, figsize=(12, 6))
        axes = np.array([[axes[0], axes[1]]])
    else:
        fig, axes = plt.subplots(n_total, 2, figsize=(12, 6 * n_total))
    
    for i, (_, row) in enumerate(df_bad.iterrows()):
        axes[i, 0].imshow(cv2.cvtColor(row['crop'], cv2.COLOR_BGR2RGB))
        axes[i, 0].set_title(f"Мусор: {row['confidence']:.1f}%")
        axes[i, 0].axis('off')
    
    for i, (_, row) in enumerate(df_good.iterrows()):
        axes[i, 1].imshow(cv2.cvtColor(row['crop'], cv2.COLOR_BGR2RGB))
        axes[i, 1].set_title(f"Хороший: {100 - row['confidence']:.1f}%")
        axes[i, 1].axis('off')
    
    for i in range(n_good, n_total):
        axes[i, 1].axis('off')
    for i in range(n_bad, n_total):
        axes[i, 0].axis('off')
    
    plt.tight_layout()
    plt.show()

print(f"Мусор: {len(df_bad)}, Хороших: {len(df_good)}")

In [ ]:
# === ВИЗУАЛИЗАЦИЯ: Оригинальные bbox на 5 фреймах (по одному из каждого сегмента) ===
print("=== ОРИГИНАЛЬНЫЕ BBOX НА ФРЕЙМАХ (5 кадров) ===")

# Берём по одному фрейму из каждого сегмента (первый кадр сегмента)
frames_to_show = []
for sid, frames in enumerate(segment_frames):
    if frames:
        frames_to_show.append(frames[0])

cap = cv2.VideoCapture(str(video_path))

for fi in frames_to_show:
    cap.set(cv2.CAP_PROP_POS_FRAMES, fi)
    ret, fr = cap.read()
    if not ret:
        continue
    
    timestamp_ms = int(fi / fps * 1000)
    
    # Рисуем bbox на оригинальном фрейме (ДО поворота и дисторсии)
    vis = fr.copy()
    boxes = original_bbox_by_timestamp.get(timestamp_ms, [])
    
    for track_id, bbox in boxes:
        x1, y1, x2, y2 = bbox
        cv2.rectangle(vis, (x1, y1), (x2, y2), (0, 255, 0), 3)
        cv2.putText(vis, f"ID:{track_id}", (x1, max(y1 - 10, 25)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
    
    # Конвертируем BGR → RGB для matplotlib
    rgb = cv2.cvtColor(vis, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(16, 9))
    plt.imshow(rgb)
    ts_sec = timestamp_ms / 1000.0
    plt.title(f"Frame: {fi} | Time: {ts_sec:.1f}s | Boxes: {len(boxes)}", fontsize=14)
    plt.axis('off')
    plt.show()

cap.release()
print(f"Показано {len(frames_to_show)} фреймов с bbox")

In [ ]:
import gc, torch
del yolo, classifier
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()

In [ ]:
# Загрузка 
vlm_model, vlm_processor = load_vlm_model(str(root / 'VLM_MODULE' / 'AVITO'), config_name='High Quality 8-bit')

# crop_array from best into df_crops
crop_map = {}
for tid, candidates in best.items():
    for rank, c in enumerate(candidates):
        crop_map[(tid, rank + 1)] = c.crop

df_crops['crop_array'] = df_crops.apply(
    lambda r: crop_map.get((r['SYS_track_id'], r['SYS_rank']), None), axis=1)

df_vlm = vlm_predict_crops(df_crops, vlm_model, vlm_processor)

In [ ]:
df_vlm = pd.read_excel('vlm_new2.xlsx') 

In [ ]:

df_vlm_match = df_vlm.rename(columns={'product_name': 'ocr_text'})
df_vlm_match = find_top5_matches(df_vlm_match, ocr_col='ocr_text')
display(df_vlm_match)